In [45]:
from dbnomics import fetch_series
import pandas as pd
import os

In [6]:
df_cpi = fetch_series('IMF/CPI/M.PH.PCPIT_IX')
df_cpi.isnull().sum()

@frequency           0
provider_code        0
dataset_code         0
dataset_name         0
series_code          0
series_name          0
original_period      0
period               0
original_value       0
value              185
FREQ                 0
REF_AREA             0
INDICATOR            0
Frequency            0
Reference Area       0
Indicator            0
dtype: int64

In [7]:
df_cpi

,@frequency,provider_code,dataset_code,dataset_name,series_code,series_name,original_period,period,original_value,value,FREQ,REF_AREA,INDICATOR,Frequency,Reference Area,Indicator
0,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2010-01,2010-01-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
1,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2010-02,2010-02-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
2,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2010-03,2010-03-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
3,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2010-04,2010-04-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
4,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2010-05,2010-05-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2025-01,2025-01-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
181,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2025-02,2025-02-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
182,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2025-03,2025-03-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport
183,monthly,IMF,CPI,Consumer Price Index (CPI),M.PH.PCPIT_IX,Monthly – Philippines – Transport,2025-04,2025-04-01,NA,None,M,PH,PCPIT_IX,Monthly,Philippines,Transport


In [8]:
print(df_cpi[df_cpi['value'].notnull()].tail(10))

Empty DataFrame
Columns: [@frequency, provider_code, dataset_code, dataset_name, series_code, series_name, original_period, period, original_value, value, FREQ, REF_AREA, INDICATOR, Frequency, Reference Area, Indicator]
Index: []


In [21]:
from dbnomics import fetch_series
import pandas as pd

# This is the Headline CPI (All Items) for the Philippines - 2018 base year
# It is much more likely to have actual 'value' data than the sub-transport index
df_cpi = fetch_series('IMF/CPI/M.PH.PCPI_IX')

# 1. Drop the empty rows that are causing the "Empty DataFrame" look
df_cpi = df_cpi.dropna(subset=['value'])

# 2. Filter for your model's timeline
df_cpi['period'] = pd.to_datetime(df_cpi['period'])
df_cpi = df_cpi[df_cpi['period'] >= '2015-01-01']

print(f"Success! Retrieved {len(df_cpi)} monthly data points.")
df_cpi[['period', 'value']]

Success! Retrieved 125 monthly data points.


,period,value
780,2015-01-01,90.981245
781,2015-02-01,91.066433
782,2015-03-01,90.981245
783,2015-04-01,91.151621
784,2015-05-01,91.066433
...,...,...
900,2025-01-01,128.400000
901,2025-02-01,128.100000
902,2025-03-01,127.800000
903,2025-04-01,127.300000


In [43]:
# select and rename
df_cpi_fil = df_cpi[['period', 'value']].rename(columns={'period': 'date'})

# convert to datetime WHILE it is still a column
df_cpi_fil['date'] = pd.to_datetime(df_cpi_fil['date'])

# set as the index
df_cpi_fil = df_cpi_fil.set_index('date')

# resample to weekly
df_weekly_cpi = df_cpi_fil.resample('W-FRI').ffill()
df_weekly_cpi.index = df_weekly_cpi.index - pd.Timedelta(days=4)
df_weekly_cpi.index.name = "week_start"

# rename
df_weekly_cpi = df_weekly_cpi.rename(columns={'value': 'cpi_value'})

In [44]:
df_weekly_cpi

,cpi_value
week_start,
2014-12-29,90.981245
2015-01-05,90.981245
2015-01-12,90.981245
2015-01-19,90.981245
2015-01-26,90.981245
...,...
2025-03-31,127.300000
2025-04-07,127.300000
2025-04-14,127.300000


In [46]:
os.makedirs("data/semi-cleaned", exist_ok=True)

In [47]:
df_weekly_cpi.to_parquet("data/semi-cleaned/full_cpi.parquet")